# Task 3: Feature Engineering

**BCG X Data Science Job Simulation on Forage - PowerCo churn case**

> **Simulation disclosure:** This notebook documents work completed as part of the **BCG X Data Science Job Simulation on Forage**. It is an educational job simulation and does not represent employment, an internship, or client work performed for BCG X or PowerCo.

## Objective
Transform the cleaned customer and pricing data into model-ready variables that represent price movement, contract timing, consumption behavior, customer value, product engagement, and forecast relationships.


In [1]:
import pandas as pd

## 1. Load prepared customer and pricing data

In [2]:
df = pd.read_csv('../data/processed/clean_data_after_eda.csv')
df["date_activ"] = pd.to_datetime(df["date_activ"], format='%Y-%m-%d')
df["date_end"] = pd.to_datetime(df["date_end"], format='%Y-%m-%d')
df["date_modif_prod"] = pd.to_datetime(df["date_modif_prod"], format='%Y-%m-%d')
df["date_renewal"] = pd.to_datetime(df["date_renewal"], format='%Y-%m-%d')

In [3]:
df.head(3)

,id,channel_sales,cons_12m,cons_gas_12m,cons_last_month,date_activ,date_end,date_modif_prod,date_renewal,forecast_cons_12m,forecast_cons_year,forecast_discount_energy,forecast_meter_rent_12m,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_pow_off_peak,has_gas,imp_cons,margin_gross_pow_ele,margin_net_pow_ele,nb_prod_act,net_margin,num_years_antig,origin_up,pow_max,var_year_price_off_peak_var,var_year_price_peak_var,var_year_price_mid_peak_var,var_year_price_off_peak_fix,var_year_price_peak_fix,var_year_price_mid_peak_fix,var_year_price_off_peak,var_year_price_peak,var_year_price_mid_peak,var_6m_price_off_peak_var,var_6m_price_peak_var,var_6m_price_mid_peak_var,var_6m_price_off_peak_fix,var_6m_price_peak_fix,var_6m_price_mid_peak_fix,var_6m_price_off_peak,var_6m_price_peak,var_6m_price_mid_peak,churn
0,24011ae4ebbe3035111d65fa7c15bc57,foosdfpfkusacimwkcsosbicdxkicaua,0,54946,0,2013-06-15,2016-06-15,2015-11-01,2015-06-23,0.00,0,0.0,1.78,0.114481,0.098142,40.606701,t,0.0,25.44,25.44,2,678.99,3,lxidpiddsbxsbosboudacockeimpuepw,43.648,0.000061,2.627605e-05,0.00044,1.102785,49.550703,22.022535,1.102846,4.955073e+01,22.022975,0.000131,4.100838e-05,0.000908,2.086294,99.530517,44.235794,2.086425,9.953056e+01,44.236702,1
1,d29c2c54acc38ff3c0614d0a653813dd,MISSING,4660,0,0,2009-08-21,2016-08-30,2009-08-21,2015-08-31,189.95,0,0.0,16.27,0.145711,0.000000,44.311378,f,0.0,16.38,16.38,1,18.89,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.800,0.000005,6.089453e-04,0.00000,0.006465,0.000000,0.000000,0.006470,6.089453e-04,0.000000,0.000003,1.217891e-03,0.000000,0.009482,0.000000,0.000000,0.009485,1.217891e-03,0.000000,0
2,764c75f661154dac3a6c254cd082ea7d,foosdfpfkusacimwkcsosbicdxkicaua,544,0,0,2010-04-16,2016-04-16,2010-04-16,2015-04-17,47.96,0,0.0,38.72,0.165794,0.087899,44.311378,f,0.0,28.60,28.60,1,6.60,6,kamkkxfxxuwbdslkwifmmcsiusiuosws,13.856,0.000006,2.558511e-07,0.00000,0.007662,0.000000,0.000000,0.007668,2.558511e-07,0.000000,0.000004,9.450150e-08,0.000000,0.000000,0.000000,0.000000,0.000004,9.450150e-08,0.000000,0


In [4]:
price_df = pd.read_csv('../data/raw/price_data.csv')
price_df["price_date"] = pd.to_datetime(price_df["price_date"], format='%Y-%m-%d')
price_df.head()

,id,price_date,price_off_peak_var,price_peak_var,price_mid_peak_var,price_off_peak_fix,price_peak_fix,price_mid_peak_fix
0,038af19179925da21a25619c5a24b745,2015-01-01,0.151367,0.0,0.0,44.266931,0.0,0.0
1,038af19179925da21a25619c5a24b745,2015-02-01,0.151367,0.0,0.0,44.266931,0.0,0.0
2,038af19179925da21a25619c5a24b745,2015-03-01,0.151367,0.0,0.0,44.266931,0.0,0.0
3,038af19179925da21a25619c5a24b745,2015-04-01,0.149626,0.0,0.0,44.266931,0.0,0.0
4,038af19179925da21a25619c5a24b745,2015-05-01,0.149626,0.0,0.0,44.266931,0.0,0.0


## 2. Baseline price-change feature

Recreate the simulation baseline feature measuring the change in off-peak energy and power prices between the first and last observed months for each customer.

In [5]:
# Group off-peak prices by companies and month
monthly_price_by_id = price_df.groupby(['id', 'price_date']).agg({'price_off_peak_var': 'mean', 'price_off_peak_fix': 'mean'}).reset_index()

# Get january and december prices
jan_prices = monthly_price_by_id.groupby('id').first().reset_index()
dec_prices = monthly_price_by_id.groupby('id').last().reset_index()

# Calculate the difference
diff = pd.merge(dec_prices.rename(columns={'price_off_peak_var': 'dec_1', 'price_off_peak_fix': 'dec_2'}), jan_prices.drop(columns='price_date'), on='id')
diff['offpeak_diff_dec_january_energy'] = diff['dec_1'] - diff['price_off_peak_var']
diff['offpeak_diff_dec_january_power'] = diff['dec_2'] - diff['price_off_peak_fix']
diff = diff[['id', 'offpeak_diff_dec_january_energy','offpeak_diff_dec_january_power']]
diff.head()

,id,offpeak_diff_dec_january_energy,offpeak_diff_dec_january_power
0,0002203ffbb812588b632b9e628cc38d,-0.006192,0.162916
1,0004351ebdd665e6ee664792efc4fd13,-0.004104,0.177779
2,0010bcc39e42b3c2131ed2ce55246e3c,0.050443,1.500000
3,0010ee3855fdea87602a5b7aba8e42de,-0.010018,0.162916
4,00114d74e963e47177db89bc70108537,-0.003994,-0.000001


In [6]:
df = pd.merge(
    df,
    diff,
    on='id',
    how='left'
)

df.shape

(14606, 46)

## 3. Contract timing and tenure features

Represent customer tenure and remaining contract duration relative to the modeling reference date.

In [7]:
reference_date = pd.Timestamp('2016-01-01')

df['tenure_years'] = (
    (reference_date - df['date_activ']).dt.days / 365.25
)

df['months_to_end'] = (
    (df['date_end'] - reference_date).dt.days / 30.44
)

df[['tenure_years', 'months_to_end']].head()

,tenure_years,months_to_end
0,2.546201,5.453351
1,6.362765,7.950066
2,5.711157,3.482260
3,5.757700,2.923784
4,5.965777,2.168200


## 4. Consumption behavior

Create features that compare recent consumption with longer-term usage patterns.

In [8]:
# Average monthly consumption based on the last 12 months
df['avg_monthly_cons'] = df['cons_12m'] / 12

# Difference between recent consumption and historical monthly average
df['cons_change_last_month'] = (
    df['cons_last_month'] - df['avg_monthly_cons']
)

df[['cons_12m', 'cons_last_month',
    'avg_monthly_cons', 'cons_change_last_month']].head()

,cons_12m,cons_last_month,avg_monthly_cons,cons_change_last_month
0,0,0,0.000000,0.000000
1,4660,0,388.333333,-388.333333
2,544,0,45.333333,-45.333333
3,1584,0,132.000000,-132.000000
4,4425,526,368.750000,157.250000


## 5. Margin and customer-value features

Capture differences between gross and net power margins and normalize margin by consumption.

In [9]:
# Difference between gross and net power margins
df['margin_difference'] = (
    df['margin_gross_pow_ele'] - df['margin_net_pow_ele']
)

# Net margin per unit of annual consumption
df['net_margin_per_cons'] = (
    df['net_margin'] / (df['cons_12m'] + 1)
)

df[['margin_gross_pow_ele', 'margin_net_pow_ele',
    'margin_difference', 'net_margin_per_cons']].head()

,margin_gross_pow_ele,margin_net_pow_ele,margin_difference,net_margin_per_cons
0,25.44,25.44,0.0,678.990000
1,16.38,16.38,0.0,0.004053
2,28.60,28.60,0.0,0.012110
3,30.22,30.22,0.0,0.016063
4,44.91,44.91,0.0,0.010840


## 6. Forecast pricing relationship

Inspect available price variables and create the peak versus off-peak forecast price difference.

In [10]:
[col for col in df.columns if 'price' in col]

['forecast_price_energy_off_peak',
 'forecast_price_energy_peak',
 'forecast_price_pow_off_peak',
 'var_year_price_off_peak_var',
 'var_year_price_peak_var',
 'var_year_price_mid_peak_var',
 'var_year_price_off_peak_fix',
 'var_year_price_peak_fix',
 'var_year_price_mid_peak_fix',
 'var_year_price_off_peak',
 'var_year_price_peak',
 'var_year_price_mid_peak',
 'var_6m_price_off_peak_var',
 'var_6m_price_peak_var',
 'var_6m_price_mid_peak_var',
 'var_6m_price_off_peak_fix',
 'var_6m_price_peak_fix',
 'var_6m_price_mid_peak_fix',
 'var_6m_price_off_peak',
 'var_6m_price_peak',
 'var_6m_price_mid_peak']

In [11]:
df['forecast_price_energy_diff'] = (
    df['forecast_price_energy_peak']
    - df['forecast_price_energy_off_peak']
)

df[[
    'forecast_price_energy_off_peak',
    'forecast_price_energy_peak',
    'forecast_price_energy_diff'
]].head()

,forecast_price_energy_off_peak,forecast_price_energy_peak,forecast_price_energy_diff
0,0.114481,0.098142,-0.016339
1,0.145711,0.000000,-0.145711
2,0.165794,0.087899,-0.077895
3,0.146694,0.000000,-0.146694
4,0.116900,0.100015,-0.016885


## 7. Product and service indicators

Encode gas-service status and whether a customer holds multiple active products.

In [12]:
df['has_gas_binary'] = df['has_gas'].map({
    't': 1,
    'f': 0
})

df[['has_gas', 'has_gas_binary']].head()

,has_gas,has_gas_binary
0,t,1
1,f,0
2,f,0
3,f,0
4,f,0


In [13]:
df['has_multiple_products'] = (df['nb_prod_act'] > 1).astype(int)

df[['nb_prod_act', 'has_multiple_products']].head()

,nb_prod_act,has_multiple_products
0,2,1
1,1,0
2,1,0
3,1,0
4,1,0


## 8. Renewal and product-change timing

Represent proximity to renewal and whether the customer has experienced a product change since activation.

In [14]:
df['days_to_renewal'] = (
    df['date_renewal'] - pd.Timestamp('2016-01-01')
).dt.days

df[['date_renewal', 'days_to_renewal']].head()

,date_renewal,days_to_renewal
0,2015-06-23,-192
1,2015-08-31,-123
2,2015-04-17,-259
3,2015-03-31,-276
4,2015-03-09,-298


In [15]:
df['days_to_product_change'] = (
    df['date_modif_prod'] - df['date_activ']
).dt.days

df[['date_activ', 'date_modif_prod', 'days_to_product_change']].head()

,date_activ,date_modif_prod,days_to_product_change
0,2013-06-15,2015-11-01,869
1,2009-08-21,2009-08-21,0
2,2010-04-16,2010-04-16,0
3,2010-03-30,2010-03-30,0
4,2010-01-13,2010-01-13,0


In [16]:
df['has_product_changed'] = (
    df['days_to_product_change'] > 0
).astype(int)

df[['days_to_product_change', 'has_product_changed']].head()

,days_to_product_change,has_product_changed
0,869,1
1,0,0
2,0,0
3,0,0
4,0,0


## 9. Additional predictive features

Add consumption ratios, forecast-versus-actual measures, commercial-value ratios, aggregate price variation, forecast price ratios, and calendar features.

In [17]:
# 1. Consumption ratios
# Compare recent consumption with average monthly consumption
df['recent_cons_ratio'] = (
    df['cons_last_month'] /
    (df['avg_monthly_cons'] + 1)
)

# Gas consumption as a proportion of total annual consumption
df['gas_cons_ratio'] = (
    df['cons_gas_12m'] /
    (df['cons_12m'] + df['cons_gas_12m'] + 1)
)


# 2. Forecast vs actual consumption
# Difference between forecast annual consumption and actual consumption
df['forecast_actual_cons_diff'] = (
    df['forecast_cons_12m'] - df['cons_12m']
)

# Ratio between forecast and actual consumption
df['forecast_actual_cons_ratio'] = (
    df['forecast_cons_12m'] /
    (df['cons_12m'] + 1)
)


# 3. Customer value / engagement
# Net margin per active product
df['margin_per_product'] = (
    df['net_margin'] /
    (df['nb_prod_act'] + 1)
)

# Subscribed power relative to annual electricity consumption
df['power_per_cons'] = (
    df['pow_max'] /
    (df['cons_12m'] + 1)
)


# 4. Price-change features
# Combine annual price variation across off-peak, peak and mid-peak
df['total_year_price_variation'] = (
    df['var_year_price_off_peak'] +
    df['var_year_price_peak'] +
    df['var_year_price_mid_peak']
)

# Combine six-month price variation
df['total_6m_price_variation'] = (
    df['var_6m_price_off_peak'] +
    df['var_6m_price_peak'] +
    df['var_6m_price_mid_peak']
)


# 5. Forecast price relationship
# Ratio of peak to off-peak forecast energy price
df['forecast_peak_offpeak_ratio'] = (
    df['forecast_price_energy_peak'] /
    (df['forecast_price_energy_off_peak'] + 1e-6)
)


# 6. Extract useful information from dates
df['activation_month'] = df['date_activ'].dt.month
df['renewal_month'] = df['date_renewal'].dt.month
df['product_modification_month'] = df['date_modif_prod'].dt.month


# Check newly created features
new_features = [
    'recent_cons_ratio',
    'gas_cons_ratio',
    'forecast_actual_cons_diff',
    'forecast_actual_cons_ratio',
    'margin_per_product',
    'power_per_cons',
    'total_year_price_variation',
    'total_6m_price_variation',
    'forecast_peak_offpeak_ratio',
    'activation_month',
    'renewal_month',
    'product_modification_month'
]

df[new_features].head()

,recent_cons_ratio,gas_cons_ratio,forecast_actual_cons_diff,forecast_actual_cons_ratio,margin_per_product,power_per_cons,total_year_price_variation,total_6m_price_variation,forecast_peak_offpeak_ratio,activation_month,renewal_month,product_modification_month
0,0.000000,0.999982,0.00,0.000000,226.330,43.648000,72.676550,145.853685,0.857270,6,6,11
1,0.000000,0.000000,-4470.05,0.040753,9.445,0.002961,0.007079,0.010703,0.000000,8,8,8
2,0.000000,0.000000,-496.04,0.088000,3.300,0.025424,0.007668,0.000004,0.530167,4,4,4
3,0.000000,0.000000,-1343.96,0.151445,12.730,0.008328,0.006470,0.000003,0.000000,3,3,3
4,1.422583,0.000000,-3979.25,0.100712,23.990,0.004474,0.008273,0.000014,0.855553,1,3,1


## 10. Validation

In [18]:
print("Final dataset shape:", df.shape)
print("\nTotal missing values:", df.isnull().sum().sum())
print("\nDuplicate customer IDs:", df['id'].duplicated().sum())

print("\nInfinite values in numeric columns:")
numeric_df = df.select_dtypes(include='number')
print((numeric_df == float('inf')).sum().sum() +
      (numeric_df == float('-inf')).sum().sum())

print("\nTotal features:", df.shape[1])

Final dataset shape: (14606, 70)

Total missing values: 0

Duplicate customer IDs: 0

Infinite values in numeric columns:
0

Total features: 70


## 11. Feature-engineering summary

The feature-engineering stage expands the dataset with variables covering:

- annual and six-month price variation
- customer tenure and contract timing
- recent versus historical consumption
- forecast versus actual consumption
- margin and value-per-product measures
- gas-service and multi-product indicators
- renewal and product-change timing
- seasonal activation, renewal, and product-modification signals

The resulting dataset contains **14,606 customers and 70 columns**, with **0 missing values**, **0 duplicate customer IDs**, and **0 infinite numeric values** in the completed simulation run. This prepared dataset is ready for the predictive-modeling stage.
